# EXO-200 Transformer EnergyBench Evaluation

Evaluates the six saved Transformer test-prediction files with Wing's frozen EnergyBench protocol. This notebook does not load model checkpoints onto a GPU and does not rerun inference. `Rotated_energy` is attached only as evaluation metadata.

In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd
from IPython.display import Image, display

configured_root = os.environ.get('EXO200_TRANSFORMER_PROJECT_ROOT')
candidate_roots = []
if configured_root:
    candidate_roots.append(Path(configured_root).expanduser())
candidate_roots.extend([
    Path.cwd() / 'exo200_detector',
    Path.cwd(),
    Path.cwd().parent / 'exo200_detector',
    Path.cwd().parent,
    Path.cwd().parent.parent / 'exo200_detector',
])
PROJECT_ROOT = next(
    (candidate.resolve() for candidate in candidate_roots
     if (candidate / 'exo_transformer').is_dir()
     and (candidate / 'exobench').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Could not locate exo200_detector; set EXO200_TRANSFORMER_PROJECT_ROOT.'
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from exobench import DataConfig
from exo_transformer import (
    RUN_IDS, evaluate_transformer_runs, save_comparison_plots,
)

print('Project:', PROJECT_ROOT)
print('EXOBench:', Path(sys.modules['exobench'].__file__).resolve())
print('Adapter:', Path(sys.modules['exo_transformer.energy_aware'].__file__).resolve())

In [ ]:
DATA_ROOT = Path(os.environ.get(
    'EXO200_BENCH_DATA',
    str(PROJECT_ROOT.parent / 'data' / 'EXO-200'),
)).expanduser()
OUTPUT_ROOT = Path(os.environ.get(
    'EXO200_OUTPUT_ROOT',
    str(PROJECT_ROOT / 'results' / 'transformer_official_v1'),
)).expanduser()
ENERGYBENCH_SOURCE = Path(os.environ.get(
    'EXO200_ENERGYBENCH_SOURCE',
    str(PROJECT_ROOT / 'frozen_energybench'),
)).expanduser()
wing_summary_value = os.environ.get(
    'EXO200_WING_ENERGYBENCH_SUMMARY', ''
).strip()
WING_SUMMARY = (
    Path(wing_summary_value).expanduser() if wing_summary_value else None
)
REQUIRE_ALL_SIX = os.environ.get(
    'EXO200_REQUIRE_ALL_EVALUATIONS', '0'
).strip().lower() in {'1', 'true', 'yes'}

data_config = DataConfig(
    data_root=DATA_ROOT,
    validation_fraction=0.10,
    baseline_samples=200,
    classification_amplitude_normalization=True,
    seed=42,
)
print('Dataset:', DATA_ROOT)
print('Transformer outputs:', OUTPUT_ROOT)
print('Frozen EnergyBench:', ENERGYBENCH_SOURCE)
print('Expected models:', len(RUN_IDS))

## Run the frozen evaluation

This reads every completed `predictions.npz`, reconstructs the official test metadata once, verifies exact label ordering, and delegates all metrics to Wing's `run_evaluation`.

In [ ]:
results, statuses, reports = evaluate_transformer_runs(
    data_config=data_config,
    output_root=OUTPUT_ROOT,
    energybench_source=ENERGYBENCH_SOURCE,
)
display(statuses)

completed = int((statuses['status'] == 'complete').sum())
print(f'Completed evaluations: {completed}/{len(RUN_IDS)}')
if REQUIRE_ALL_SIX and completed != len(RUN_IDS):
    missing = statuses.loc[statuses['status'] != 'complete', 'run_id'].tolist()
    raise RuntimeError(f'Official evaluation is incomplete: {missing}')

In [ ]:
DISPLAY_COLUMNS = [
    'tokenization', 'position_encoding',
    'best_validation_auc', 'inclusive_auc', 'common_support_auc',
    'energy_matched_auc', 'shortcut_gap', 'matched_coverage',
    'energy_independence_score', 'worst_energy_independence_score',
    'epochs_completed', 'minutes_per_epoch', 'matched_auc_status',
]
if results.empty:
    print('No completed Transformer runs are available yet.')
else:
    display_table = results[DISPLAY_COLUMNS].copy()
    numeric_columns = [
        name for name in DISPLAY_COLUMNS
        if name not in {'tokenization', 'position_encoding', 'matched_auc_status'}
    ]
    display_table[numeric_columns] = display_table[numeric_columns].apply(
        pd.to_numeric, errors='coerce'
    )
    display(
        display_table
        .style.format({
            'best_validation_auc': '{:.6f}',
            'inclusive_auc': '{:.6f}',
            'common_support_auc': '{:.6f}',
            'energy_matched_auc': '{:.6f}',
            'shortcut_gap': '{:.6f}',
            'matched_coverage': '{:.3f}',
            'energy_independence_score': '{:.6f}',
            'worst_energy_independence_score': '{:.6f}',
            'minutes_per_epoch': '{:.2f}',
        })
        .background_gradient(subset=['energy_matched_auc'], cmap='Blues')
    )
    print('Higher AUC and energy-independence scores are better.')

## Protocol and structural checks

In [ ]:
if reports:
    evaluation_fingerprints = {
        report['evaluation_fingerprint'] for report in reports.values()
    }
    protocol_fingerprints = {
        report['protocol_fingerprint'] for report in reports.values()
    }
    assert len(evaluation_fingerprints) == 1
    assert len(protocol_fingerprints) == 1
    print('Evaluation fingerprint:', next(iter(evaluation_fingerprints)))
    print('Protocol fingerprint:', next(iter(protocol_fingerprints)))

if reports and WING_SUMMARY is not None and WING_SUMMARY.is_file():
    wing = json.loads(WING_SUMMARY.read_text())
    wing_report = next(iter(wing['models'].values()))
    transformer_report = next(iter(reports.values()))
    checks = {
        'classification sections': (
            set(wing_report['classification'])
            == set(transformer_report['classification'])
        ),
        'classification aggregate fields': (
            set(wing_report['classification']['aggregates'])
            == set(transformer_report['classification']['aggregates'])
        ),
        'energy-dependence fields': (
            set(wing_report['energy_dependence'])
            == set(transformer_report['energy_dependence'])
        ),
        'protocol fingerprint': (
            wing_report['protocol_fingerprint']
            == transformer_report['protocol_fingerprint']
        ),
    }
    display(pd.DataFrame(
        [{'check': name, 'passed': passed} for name, passed in checks.items()]
    ))
    if not all(checks.values()):
        raise RuntimeError('Transformer and Wing EnergyBench structures differ')
elif reports:
    print('Wing summary not configured; skipped optional structural comparison.')

## Comparison plots

In [ ]:
plot_paths = save_comparison_plots(
    results, reports, OUTPUT_ROOT / 'energybench_transformer_plots'
)
for path in plot_paths:
    print(path)
    display(Image(filename=str(path)))

## Saved outputs

The combined CSV and JSON are sufficient for later comparison with Wing's four specialized models. Per-model directories retain the canonical EnergyBench prediction bundle, provenance, complete report, and plots.

In [ ]:
for path in [
    OUTPUT_ROOT / 'energybench_transformer_results.csv',
    OUTPUT_ROOT / 'energybench_transformer_summary.json',
]:
    print(path, 'exists=' + str(path.is_file()))